In [1]:
from deeplatent import Corpus, GTM, generate_documents
from sklearn.feature_extraction.text import CountVectorizer

df_theta, df, topic_words, true_lambda, true_label_coeffs = generate_documents(
    num_docs=100000,
    num_topics=3,
    vocab_size=500,
    min_words=100,
    max_words=100,
    num_covs=2,
    num_languages=1,
    doc_topic_prior="logistic_normal",
    label_type="regression",
    random_seed=1234
)

In [ ]:
# ---- 1. Fit vectorizer on training set ----
vectorizer = CountVectorizer()  
vectorizer.fit(df["doc_clean_0"])

# ---- 2. Define modalities using this vectorizer ----
modalities = {
    "text": {
        "column": "doc_clean_0",
        "views": {
            "bow": {
                "type": "bow",
                "vectorizer": vectorizer
            }
        }
    }
}

# ---- 3. Create GTMCorpus datasets ----
train_dataset = Corpus(
    df,
    modalities=modalities,
    labels={"label": {"column": "label", "type": "regression"}},
    prevalence="~cov_1 + cov_2"
)

In [ ]:
# Train the model

encoder_args = {
    "text_bow": {
        "hidden_dims": [128,64,32],
        "activation": "relu",
        "bias": True,
        "dropout": 0.0
    }
}

decoder_args = {
    "text_bow": {
        "hidden_dims": [],
        "activation": "relu",
        "bias": True,
        "dropout": 0.0
    }
}

# Per-label predictor configuration
predictor_args = {
    "label": {
        "hidden_dims": [],
        "activation": "relu",
        "bias": False,
        "dropout": 0.0,
        "loss_weight": 1.0
    }
}

optim_args = {
    "main": {
        "lr": 1e-3, 
        "weight_decay": 0.0  # L2 penalty for main parameters 
    }
}

tm = GTM(
    train_dataset, 
    ae_type="vae",
    vi_type="iaf",
    update_prior=True,
    optim_args=optim_args,
    encoder_args=encoder_args,
    decoder_args=decoder_args,
    predictor_args=predictor_args,
    n_topics=3,
    batch_size=100,
    w_pred_loss=1,
    w_prior=0.01,
    print_every_n_steps=1000,
    num_steps=10000
    #kl_annealing_start=1000,
    #kl_annealing_end=10000
)

In [13]:
#tm.w_prior=1
#tm.num_steps = tm.steps + 1000  # Train N more steps
#tm.train(train_dataset)

Step  11000	Mean Training Loss:5.5184956
Rec Loss:5.1146784
Divergence Loss:0.1089277
Pred Loss:0.2948892



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import linear_sum_assignment

full_dataset = Corpus(
    df,
    modalities=modalities,
    labels={"label": {"column": "label", "type": "regression"}},
    prevalence="~cov_1 + cov_2"
)
estimated_doc_topics = tm.get_doc_topic_distribution(full_dataset, num_samples=30)

K = estimated_doc_topics.shape[1]
corr_matrix = np.zeros((K, K))

for i in range(K):
    for j in range(K):
        corr_matrix[i, j] = np.corrcoef(
            estimated_doc_topics[:, i],
            df_theta[f"Topic{j}"]
        )[0, 1]

# maximize absolute correlation
row_ind, col_ind = linear_sum_assignment(-np.abs(corr_matrix))

# reorder df_theta based on optimal assignment
df_theta_aligned = df_theta[[f"Topic{j}" for j in col_ind]]
df_theta_aligned.columns = [f"Topic{i}" for i in range(K)]

print("Topic alignment permutation:", col_ind)
print("Correlation matrix:\n", corr_matrix)

fig, axs = plt.subplots(1, 3, figsize=(18, 6))

for i in range(3):
    x = estimated_doc_topics[:, i]
    y = df_theta_aligned[f"Topic{i}"]
    
    axs[i].scatter(x, y, s=1, alpha=0.1)
    
    coefficients = np.polyfit(x, y, 1)
    fit = np.poly1d(coefficients)
    axs[i].plot(x, fit(x), alpha=0.8)
    
    axs[i].set_xlabel('Estimates', fontsize=15)
    axs[i].set_ylabel('True Value', fontsize=15)
    axs[i].set_title(f'Topic {i}', fontsize=15)

    corr_coeff = np.corrcoef(x, y)[0, 1]
    axs[i].annotate(
        f'Correlation: {corr_coeff:.2f}',
        xy=(0.55, 0.3), xycoords='axes fraction',
        fontsize=12, verticalalignment='top', horizontalalignment='left',
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5')
    )

plt.tight_layout()
plt.show()

In [ ]:
print("True label_coeffs:\n", true_label_coeffs)

print('Estimated weights:')
print(tm.predictor.predictors["label"].neural_net["pred_0"].weight.detach().cpu())

In [ ]:
# get_predictions now returns a dict mapping label names to predictions
predictions = tm.get_predictions(full_dataset, to_numpy=False, num_samples=30)
predictions["label"][0:10]

In [8]:
print(df['label'].iloc[0:10])

0   -1.758370
1   -1.710073
2   -1.449654
3   -1.049296
4   -0.798403
5   -0.139587
6   -0.849571
7   -1.282447
8   -0.469033
9   -1.296467
Name: label, dtype: float64
